# EXP-019 발언자 머리 줄 전환의 원인 분석

EXP-019(`speakers.mode=head`, 프롬프트 v3 → v4, 기준·후보 모두 j_v11)의 원인 분석이다. 판정은 보류(종합 지표 +0.014, 관련성 안전 조건 실패)이고 기준 전환은 판정과 무관하다(D-63 결정 2). 충실성 상승(+0.052)과 관련성 하락(−0.020)이 어디서 왔는지 질의 단위로 나눠 본다. 새 API 호출 없이 생성 기록과 judge 결과만 읽는다. 질의·답변·근거 원문은 출력하지 않는다.


In [1]:
import re
from collections import Counter
from pathlib import Path

import pandas as pd

from rag.gen_eval.common import read_jsonl


In [2]:
ROOT = Path.cwd().parent
BASE_JUDGE = ROOT / "data/gen_eval/judge/hybrid-cond-filter-law-ms256-k20-ce-v3-dev-full-deepseek_go_mixed-j_v11.jsonl"  # 기준(EXP-018, j_v11 재채점)
CAND_JUDGE = ROOT / "data/gen_eval/judge/hybrid-cond-filter-law-ms256-k20-ce-v4-dev-full-spk-head-deepseek_go_mixed-j_v11.jsonl"  # 후보(EXP-019)
BASE_RUNS = ROOT / "data/gen_eval/runs/hybrid-cond-filter-law-ms256-k20-ce-v3-dev-full.jsonl"
CAND_RUNS = ROOT / "data/gen_eval/runs/hybrid-cond-filter-law-ms256-k20-ce-v4-dev-full-spk-head.jsonl"
MISMATCH = "발언자 불일치"
# 질의자 주장을 사실처럼 쓴 진술을 0 판정 이유에서 찾는 어림(notebooks/38_01과 같은 패턴)
CLAIM = re.compile(r"(질의자|위원)")
CLAIM_KIND = re.compile(r"(확인|사실처럼|단정|주장)")
CITE = re.compile(r"\[(\d+)\]")


In [3]:
def by_qid(path: Path) -> dict:
    return {r["qid"]: r for r in read_jsonl(path)}


def speaker_names(run: dict) -> set[str]:
    """답변이 인용한 근거의 머리 줄 화자 이름 중 답변에 나온 것."""
    cited = {int(n) for n in CITE.findall(run["answer"])}
    names = {p["name"] for c in run["contexts"] if c["n"] in cited and c.get("speakers") for f in ("questioners", "answerers") for p in c["speakers"][f]}
    return {n for n in names if n and n in run["answer"]}


def zero_kind(v: dict) -> str:
    """0 판정 이유의 종류. 낱말 대조 어림이다."""
    if MISMATCH in v["reason"]:
        return "발언자 불일치"
    if CLAIM.search(v["reason"]) and CLAIM_KIND.search(v["reason"]):
        return "질의자 주장"
    return "그 밖"


## 데이터 로딩


In [4]:
base, cand = by_qid(BASE_JUDGE), by_qid(CAND_JUDGE)
base_runs, cand_runs = by_qid(BASE_RUNS), by_qid(CAND_RUNS)
print(len(base), len(cand), len(base_runs), len(cand_runs), set(base) == set(cand))


330 330 330 330 True


In [5]:
rows = []
for q in base:
    b, c = base[q], cand[q]
    rows.append({
        "qid": q,
        "f_base": (b.get("faithfulness") or {}).get("score"),
        "f_cand": (c.get("faithfulness") or {}).get("score"),
        "r_base": (b.get("relevance") or {}).get("score"),
        "r_cand": (c.get("relevance") or {}).get("score"),
        "named": bool(speaker_names(cand_runs[q])),
        "chars_base": len(base_runs[q]["answer"]),
        "chars_cand": len(cand_runs[q]["answer"]),
    })
df = pd.DataFrame(rows).set_index("qid")
df["f_diff"] = df.f_cand - df.f_base
df["r_diff"] = df.r_cand - df.r_base
print(df.shape, df.named.sum())
df[["f_diff", "r_diff"]].describe().round(3)


(330, 9) 147


,f_diff,r_diff
count,288.000,295.000
mean,0.052,-0.020
std,0.307,0.229
min,-1.000,-1.000
25%,0.000,0.000
50%,0.000,0.000
75%,0.000,0.000
max,1.000,0.500


## 충실성: 화자 이름이 나온 답변과 나오지 않은 답변

후보 답변이 인용 근거의 화자 이름을 적었는지로 나눠 충실성 차이의 몫을 본다. 기여는 그 묶음 차이의 합을 두 실행에서 계산된 전체 질의 수로 나눈 값이다.


In [6]:
f = df.dropna(subset=["f_diff"])
parts = f.groupby("named").f_diff.agg(질의="count", 평균차이="mean", 합="sum", 상승=lambda s: (s > 0).sum(), 하락=lambda s: (s < 0).sum())
parts["기여"] = parts["합"] / len(f)
print(f"전체 {len(f)}질의, 평균 차이 {f.f_diff.mean():+.3f}")
parts.round(3)


전체 288질의, 평균 차이 +0.052


,질의,평균차이,합,상승,하락,기여
named,,,,,,
False,149,-0.000,-0.035,18,18,-0.000
True,139,0.107,14.868,34,10,0.052


In [7]:
def zero_table(judged: dict) -> Counter:
    c = Counter()
    for r in judged.values():
        for v in (r.get("faithfulness") or {}).get("verdicts") or []:
            c["진술"] += 1
            if v["verdict"] == 0:
                c["0 판정"] += 1
                c[zero_kind(v)] += 1
    return c


pd.DataFrame({"기준": zero_table(base), "후보": zero_table(cand)}).fillna(0).astype(int)


,기준,후보
진술,753,746
0 판정,148,104
질의자 주장,127,63
그 밖,19,22
발언자 불일치,2,19


In [8]:
def named_statements(run: dict, r: dict) -> tuple[int, int, int]:
    """(이름이 든 진술, 그중 1, 이름이 없는 진술 중 1의 비율 계산용 수)."""
    names = speaker_names(run)
    vs = (r.get("faithfulness") or {}).get("verdicts") or []
    hit = [v for v in vs if any(n in v["statement"] for n in names)]
    return len(hit), sum(v["verdict"] for v in hit), len(vs)


tot = [named_statements(cand_runs[q], cand[q]) for q in cand]
hit, ok, all_st = map(sum, zip(*tot, strict=True))
all_ok = sum(v["verdict"] for r in cand.values() for v in (r.get("faithfulness") or {}).get("verdicts") or [])
print(f"후보 진술 {all_st}개 중 이름이 든 진술 {hit}개, 그중 1 판정 {ok}개({ok / hit:.3f})")
print(f"이름이 없는 진술 {all_st - hit}개 중 1 판정 {all_ok - ok}개({(all_ok - ok) / (all_st - hit):.3f})")


후보 진술 746개 중 이름이 든 진술 319개, 그중 1 판정 305개(0.956)
이름이 없는 진술 427개 중 1 판정 337개(0.789)


## 답변 관련성: 하락 질의

관련성 하락 질의를 화자 이름 유무, 라벨 변화, 답변 길이 변화로 나눈다.


In [9]:
r = df.dropna(subset=["r_diff"])
parts = r.groupby("named").r_diff.agg(질의="count", 평균차이="mean", 합="sum", 상승=lambda s: (s > 0).sum(), 하락=lambda s: (s < 0).sum())
parts["기여"] = parts["합"] / len(r)
print(f"전체 {len(r)}질의, 평균 차이 {r.r_diff.mean():+.3f}")
parts.round(3)


전체 295질의, 평균 차이 -0.020


,질의,평균차이,합,상승,하락,기여
named,,,,,,
False,153,-0.007,-1.0,7,8,-0.003
True,142,-0.035,-5.0,15,23,-0.017


In [10]:
moves = Counter((base[q]["relevance"]["label"], cand[q]["relevance"]["label"]) for q in r.index if r.r_diff[q] != 0)
pd.Series(moves).sort_values(ascending=False)


답함      일부        23
일부      답함        19
        답하지 않음     5
답하지 않음  일부         3
답함      답하지 않음     3
dtype: int64

In [11]:
ratio = (r.chars_cand / r.chars_base)
print("답변 길이 비(후보/기준) 중앙값:", round(ratio.median(), 2))
print(ratio.groupby(r.r_diff.pipe(lambda s: s.gt(0).map({True: "상승"}).fillna(s.lt(0).map({True: "하락"})).fillna("같음"))).median().round(2).to_string())
print("하락 질의 중 이름 답변:", int(((r.r_diff < 0) & r.named).sum()), "/", int((r.r_diff < 0).sum()))
print("전체 질의 중 이름 답변 비율:", round(r.named.mean(), 3))


답변 길이 비(후보/기준) 중앙값: 1.05
r_diff
같음    1.06
상승    1.08
하락    0.99
하락 질의 중 이름 답변: 23 / 31
전체 질의 중 이름 답변 비율: 0.481


In [12]:
down = [q for q in r.index if r.r_diff[q] < 0]
print("하락 질의의 후보 충실성 변화 평균:", round(df.loc[down, "f_diff"].mean(), 3))
print("하락 질의의 후보 부분 거부 수:", sum(cand[q]["refusal"] == "partial" for q in down), "기준:", sum(base[q]["refusal"] == "partial" for q in down))


하락 질의의 후보 충실성 변화 평균: 0.118
하락 질의의 후보 부분 거부 수: 4 기준: 6


## 가설 상한 재계산

`notebooks/38_01_생성실험순서.ipynb` 셀 3·9·14와 같은 계산을 후보 기록에 적용한다(D-63). 위 표의 질의자 주장 수는 발언자 불일치를 먼저 분리한 값이라 여기보다 작다.


In [13]:
def composite(r: dict) -> float | None:
    f = (r.get("faithfulness") or {}).get("score")
    v = (r.get("relevance") or {}).get("score")
    return None if f is None or v is None else (f + v) / 2


comp = {q: composite(r) for q, r in cand.items() if composite(r) is not None}
claims = lift = 0
h22 = set()
for q, r in cand.items():
    vs = (r.get("faithfulness") or {}).get("verdicts") or []
    hit = [v for v in vs if v["verdict"] == 0 and CLAIM.search(v["reason"]) and CLAIM_KIND.search(v["reason"])]
    claims += len(hit)
    h22 |= {q} if hit else set()
    if hit and q in comp:
        lift += len(hit) / len(vs) / 2
h18 = {q for q, r in cand.items() if any(i["outcome"] == "상황 미전달" for i in (r.get("item_eval") or {}).get("items") or [])}
print(f"H22 상한 +{lift / len(comp):.3f}, 질의자 주장 {claims}개, {len(h22)}질의(발언자 불일치 이유도 패턴에 맞으면 센다)")
print(f"H18 상한 +{sum(1 - comp[q] for q in h18 if q in comp) / len(comp):.3f}, {len(h18)}질의")


H22 상한 +0.046, 질의자 주장 71개, 48질의(발언자 불일치 이유도 패턴에 맞으면 센다)
H18 상한 +0.013, 30질의


## 정리

충실성 상승은 화자 이름을 적은 답변에서 나왔고 이름이 없는 답변의 평균은 바뀌지 않았다. 관련성 하락도 이름 답변에 몰린다. 잘못 귀속은 EXP-019 결과 절에 건수를 적었고, judge를 고치지 않고 생성 프롬프트로 해결한다(D-63 결정 11). 이 해석은 사람 대조 없는 추론이다.
